In [ ]:
import importlib
import pickle
import os
# --
import _001_PRC_extract_trajectories
importlib.reload(_001_PRC_extract_trajectories)
from _001_PRC_extract_trajectories import multi_load_kap_data, multi_load_fg_data, FG_TYPES, N_CHAINS_PER_FG, N_BEADS_PER_FG
# --
import _004_PRC_interaction_chains
importlib.reload(_004_PRC_interaction_chains)
from _004_PRC_interaction_chains import categorize_multiples
# --
import _013_PRC_clustering
importlib.reload(_013_PRC_clustering)
from _013_PRC_clustering import load_embed_save, load_reduce_cluster_save, load_cluster_trajectories_save

In [ ]:
radii = [20,30,40,50,60]
n_sites = [2, 4, 6]

In [ ]:
# Extract trajectories for Kaps

for sites in n_sites:
    for radius in radii:
        kap_results = multi_load_kap_data(
            input_rmf_path=f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_variants/{sites}_sites",
            kap_radius=radius,
            kap_amount=50,
            start_t=10000,
            end_t=50000,
            step_t=100,
            sims_range=range(1, 31),
            frames_per_file=1,
            one_frame_from_each=True
        )
        with open(f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/merged.pickle", "wb") as f:
            pickle.dump(kap_results, f)


Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores
Using 16 cores


In [ ]:
for sites in n_sites:
    for radius in radii:
        with open(f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/merged.pickle", "rb") as f:
            kap_results = pickle.load(f)
        for i in range(1, 31):
            os.makedirs(f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/{i}", exist_ok=True)
            with open(f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/{i}/10-50.pickle", "wb") as f:
                pickle.dump(kap_results[i*50:(i+1)*50, :, :], f)

In [ ]:
# Extract trajectories for FGs

for sites in n_sites:
    fg_results = multi_load_fg_data(
        input_rmf_path=f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_variants/{sites}_sites",
        fg_types=FG_TYPES,
        n_chains_per_fg=N_CHAINS_PER_FG,
        n_beads_per_fg=N_BEADS_PER_FG,
        start_t=10000,
        end_t=50000,
        step_t=100,
        sims_range=range(1, 31),
        frames_per_file=1,
        one_frame_from_each=True
    )

    for i, result in enumerate(fg_results):
        os.makedirs(f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/fgs_single_sim_coords/{i+1}", exist_ok=True)
        with open(f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/fgs_single_sim_coords/{i+1}/10-50-fgs.pickle", "wb") as f:
            pickle.dump(result, f)

In [ ]:
# Categorize trajectories to microstates (transient interactions) 

for sites in n_sites:
    for radius in radii:
        categorize_multiples(
            sim_indexes=range(1, 31),
            sim_times=["iDKDKDKDKDKDDKDKDK"],
            diffuser_coords_path_prefix=f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_variants/{sites}_sites/{radius}_radius/",
            fg_coords_path_prefix=f"/cs/labs/ravehb/roi.eliasian/NpcTransportExperiment/HS-AFM-Dataset/dataset/full_variants/{sites}_sites/fgs/",
            step=1,
            save_file_path=f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/categorized/3k.pickle",
            k=3
        )

In [ ]:
# Embed interactions to histograms

for sites in n_sites:
    for radius in radii:
        load_embed_save(
            window_size=10, # 1 us
            load_categorized_path=f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/categorized/3k.pickle",
            save_embedded_path=f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/embedded/10-50-3k.pickle",
            save_embedded_eighth_path=None
        )

In [ ]:
# Calculate mesostates

for sites in n_sites:
    for radius in radii:
        load_reduce_cluster_save(
            pca_components=74,
            n_clusters=[10, 20, 40, 80, 160, 320, 640, 1280],
            load_embedded_path=f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/embedded/10-50-3k.pickle",
            save_pca_cluster_path=f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/clusterings/10-50-3k-74pca-kmeans-#c#clusters.pickle"
        )

In [ ]:
# Cluster embedded trajectories to mesostates

for sites in n_sites:
    for radius in radii:
        load_cluster_trajectories_save(
            n_clusters=[10, 20, 40, 80, 160, 320, 640, 1280],
            load_embedded_path=f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/embedded/10-50-3k.pickle",
            load_pca_cluster_path=f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/clusterings/10-50-3k-74pca-kmeans-#c#clusters.pickle",
            save_clustered_trajectories_path=f"/cs/labs/ravehb/roi.eliasian/Master/NPC-markov/data/ntr_variants/{sites}_sites/{radius}_radius/clustered/10-50-3k-74pca-kmeans-#c#clusters.pickle"
        )